# 🔥 Advanced · LLM evaluation — lm-eval-harness

> 🔥 **Advanced / heavy lab.** Score an LLM on standard benchmarks with the de-facto evaluation harness.
>
> **Runtime → Change runtime type → T4 GPU required.** Est. **10–30 min (task-dependent)** including downloads. Built on the official **[EleutherAI/lm-evaluation-harness](https://github.com/EleutherAI/lm-evaluation-harness)** and authored to its documented recipe — **not pre-executed here** (needs a GPU + large downloads). If a step fails, see *Troubleshooting* at the bottom; pin versions as noted.

Use it to quantify a base model vs. your QLoRA / DPO fine-tune.

## Compute · storage · time

| Resource | Demo (this notebook, free Colab T4) | Full / production run |
|---|---|---|
| **GPU** | matches the model (T4 for ≤ 7B) | A100 for 13–70B |
| **Storage** | task data + model | benchmark suites a few–tens of GB |
| **Time** | `--limit 100`, 2 tasks ~ 5–15 min | full MMLU + GSM8K + … ~ 1–4 h (model-dependent) |

**Full pipeline (scale-up):** drop `--limit`, add tasks, `--num_fewshot`, batch on multi-GPU (`accelerate`).

> Rough estimates — real numbers depend on hardware, data size and library versions.

In [ ]:
!nvidia-smi -L
!pip install -q lm-eval

## 1 · Run a few tasks (limited for speed)

In [ ]:
!lm_eval --model hf \
  --model_args pretrained=Qwen/Qwen2.5-0.5B-Instruct,dtype=bfloat16 \
  --tasks arc_easy,hellaswag \
  --device cuda:0 --batch_size auto --limit 100 \
  --output_path results

## 2 · Read the scores

In [ ]:
import json, glob
f = sorted(glob.glob("results/**/*.json", recursive=True))[-1]
res = json.load(open(f))["results"]
for task, m in res.items():
    print(task, {k: round(v, 4) for k, v in m.items() if isinstance(v, float)})

## Save & persist your results
This pipeline writes its checkpoints, metrics/logs and outputs to the run/output directory shown above (e.g. `output/`, `outputs/`, `logdir/`, `experiments/`, or the trainer's `output_dir`). **Colab sessions are ephemeral** — to keep them, either mount Drive and copy the folder (`from google.colab import drive; drive.mount('/content/drive')`) or zip + download it (`import shutil; shutil.make_archive('run','zip','OUTPUT_DIR')` then `from google.colab import files; files.download('run.zip')`). The 🤗 Trainer labs also support `trainer.push_to_hub()`. To publish any output folder as a **model repo** (then group repos into a **Collection** on your profile): `from huggingface_hub import HfApi; HfApi().upload_folder(folder_path="OUTPUT_DIR", repo_id="<you>/<lab>")`.

## How this links to tracks A–D
Honest evaluation (lesson C9) applies to every track:
- **A / B / C / D** — always score against a baseline and report failure modes, not just one number.

## Next steps
- Drop `--limit` and add tasks: `mmlu`, `gsm8k`, `truthfulqa`, `winogrande`.
- Few-shot: `--num_fewshot 5`. Evaluate a fine-tune by pointing `pretrained=` at your merged model.
- Compare base vs. fine-tuned to measure your SFT/DPO gains.